In [1]:
from financial_rag.llm.router import generate_response
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

print(PROJECT_ROOT)

/Users/pushkarkamat/Desktop/financial-rag


In [2]:
from sentence_transformers import SentenceTransformer

EMBEDDING_MODEL_NAME = "BAAI/bge-large-en-v1.5"
embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)

print("Embedding model loaded.")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Embedding model loaded.


In [3]:
from qdrant_client import QdrantClient

QDRANT_PATH = PROJECT_ROOT / "data" / "processed" / "vector_store" / "qdrant"
COLLECTION_NAME = "financial_policies"

qdrant_client = QdrantClient(path=str(QDRANT_PATH))

print("Qdrant connected.")
print(qdrant_client.get_collection(COLLECTION_NAME))

Qdrant connected.
status=<CollectionStatus.GREEN: 'green'> optimizer_status=<OptimizersStatusOneOf.OK: 'ok'> warnings=None indexed_vectors_count=0 points_count=295 segments_count=1 config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1024, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, memory=None, datatype=None, multivector_config=None), shard_number=None, sharding_method=None, replication_factor=None, write_consistency_factor=None, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=None, payload=None, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=None, memory=None, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=20000, flush_interval_sec=5, max_optim

In [5]:
def retrieve_chunks(query: str, top_k: int = 5):
    """
    Convert the query into a BGE embedding and retrieve
    the most relevant chunks from Qdrant.
    """

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = qdrant_client.query_points(
        collection_name=COLLECTION_NAME,
        query=query_embedding,
        limit=top_k,
        with_payload=True,
    )

    return results.points

In [6]:
query = "What is the maximum personal loan amount that CA2 can approve?"

results = retrieve_chunks(query, top_k=5)

print(f"Retrieved chunks: {len(results)}")

Retrieved chunks: 5


In [7]:
for i, result in enumerate(results, start=1):
    print(f"\n--- RESULT {i} ---")
    print("Score:", result.score)
    print("Payload:", result.payload)


--- RESULT 1 ---
Score: 0.7521867584392447
Payload: {'text': 'Section: 4. Personal Loan Authority\n\nStandard personal loans retain the general CA1 and CA2 retail limits but remain subject to the product cap and any elevated authority triggered by risk or exception status.\n\nSection: 4. Personal Loan Authority\n\nAuthority administration treats the controlled schedule as the source of decision rights; workflow permissions are supporting evidence only. The requirements in this section apply unless a documented product rule, approved exception, or later effective instruction expressly provides otherwise.\n\nSection: 4. Personal Loan Authority\n\nControl requirements\n\nSection: 4. Personal Loan Authority\n\n• CA1 may approve standard personal loans up to ₹10 lakh.\n\nSection: 4. Personal Loan Authority\n\n• CA2 may approve standard personal loans up to the standard product cap of ₹20 lakh.\n\nSection: 4. Personal Loan Authority\n\n• CA3 and above may approve within their general moneta

---

## Create a context formatter

In [8]:
def format_context(results):
    """
    Convert retrieved Qdrant results into a readable context
    for the LLM.
    """

    context_parts = []

    for i, result in enumerate(results, start=1):
        payload = result.payload

        context_parts.append(
            f"""SOURCE {i}
Document: {payload.get('document')}
Section: {payload.get('section')}
Chunk ID: {payload.get('chunk_id')}
Content:
{payload.get('text')}
"""
        )

    return "\n\n".join(context_parts)

## Create the grounded RAG prompt

In [16]:
def build_rag_prompt(question, context):
    return f"""
You are a financial policy assistant for Northstar Financial.

Answer the user's question using ONLY the policy context provided below.

Rules:
1. Do not use outside knowledge.
2. Do not invent policy rules, limits, authorities, or exceptions.
3. If the context does not contain enough information to answer confidently, say:
   "Sorry, I could not find sufficient support for this answer in the provided policies or documents."
4. When multiple policies are relevant, consider all of them.
5. Preserve exact monetary amounts, thresholds, authority levels, and conditions.
6. Give a concise explanation.
7. Cite the relevant source document and section in your answer.

POLICY CONTEXT:
----------------
{context}
----------------

USER QUESTION:
{question}

ANSWER:
"""

## Create RAG function

In [10]:
def answer_question(question, top_k=5):
    """
    Retrieve relevant policy chunks and generate
    a grounded answer using the LLM router.
    """

    result = retrieve_chunks(question, top_k=top_k)
    context = format_context(result)

    prompt = build_rag_prompt(
        question=question,
        context=context
    )

    answer, provider = generate_response(prompt)

    return {
        "question": question,
        "answer": answer,
        "provider": provider,
        "results": results,
    }

## Test complete RAG pipeline

In [11]:
result = answer_question(
    "what is the maximum personal loan amount that CA2 can approve"
)

print("Provider:", result["provider"])
print("\nAnswer:")
print(result["answer"])

Direct use of automatic function calling (AFC) in Models.generate_content is not recommended. Instead, we recommend to use AFC in Chat.send_message. Similarly, direct use of AFC in Models.generate_content_stream is not recommended. Instead, we recommend to use AFC in Chat.send_message_stream.


Provider: gemini

Answer:
CA2 may approve standard personal loans up to a maximum of ₹20 lakh.

Sources:
*   *03_Credit_Delegated_Authority_Schedule_CDAS-007_v2.1.docx*, Section 4: Personal Loan Authority
*   *01_Credit_Risk_Policy_CRP-001_v2.0.docx*, Section 13: Credit Approval and Delegated Authority


### Test an important exception

In [12]:
result = answer_question(
    "Can a borrower with a personal-loan credit score of 630 be approved?"
)

print("Provider:", result["provider"])
print("\nAnswer:")
print(result["answer"])

Provider: gemini

Answer:
Yes, a borrower with a personal-loan credit score of 630 can be approved, provided they meet the following requirements:

*   **Exception Classification:** A score of 630 is classified as an E2 exception (Source 1, Section 7).
*   **Approval Requirement:** The application requires manual underwriting and at least CA3 approval (Source 1, Section 7; Source 2, Section 8).
*   **Eligibility:** Approval is subject to the case not being rendered ineligible due to recent severe delinquency (Source 1, Section 7).


### Test the abstention behavior

In [15]:
result = answer_question(
    "What is Northstar Financial's policy for agricultural equipment loans?"
)

print("Provider:", result["provider"])
print("\nAnswer:")
print(result["answer"])

Provider: gemini

Answer:
Sorry, I could not find sufficient support for this answer in the provided policies or document.
